# Semantic Bible: baseline

Семантический поиск по стихам Синодального перевода **без дообучения**:
эмбеддер `intfloat/multilingual-e5-small` + индекс `FAISS IndexFlatIP`.

Основа проекта: она работает ещё до fine-tuning и служит
точкой отсчёта для сравнения base vs fine-tuned.

- e5-small — сильная многоязычная модель на 384 измерения, быстрая на T4.
  Семейство e5 обучено с инструктивными префиксами, поэтому запрос кодируется
  с `query: `, а стих — с `passage: ` (без префиксов качество заметно падает).

- Нормируем векторы и берём `IndexFlatIP`: скалярное произведение нормированных
  векторов = косинусная близость. На 37 тыс. стихов точный (brute-force) индекс
  мгновенен; приближённые IVF/HNSW нужны на миллионах векторов, здесь излишни.

- Позиция вектора в индексе совпадает с `verse_id`, поэтому результат поиска
  сразу даёт id стиха в корпусе.

In [1]:
# 1. Зависимости
!pip -q install sentence-transformers faiss-cpu pyarrow

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.5/18.5 MB 36.7 MB/s eta 0:00:00


In [2]:
# 2. Google Drive для сохранения индекса между сессиями
from google.colab import drive
drive.mount('/content/drive')

import os
WORK = '/content/drive/MyDrive/semantic_bible'   # папка проекта на Drive
os.makedirs(WORK, exist_ok=True)
print('Рабочая папка:', WORK)

Mounted at /content/drive
Рабочая папка: /content/drive/MyDrive/semantic_bible


## 3. Загрузка корпуса

Грузим `corpus.parquet`. Порядок строк = `verse_id`.

In [3]:
import pandas as pd

CORPUS_PATH = f'{WORK}/corpus.parquet'
corpus = pd.read_parquet(CORPUS_PATH).reset_index(drop=True)

print('Стихов:', len(corpus))
corpus[['verse_id','reference','verse']].head(3)

Стихов: 37098


,verse_id,reference,verse
0,0,Бытие 1:1,В начале сотворил Бог небо и землю.
1,1,Бытие 1:2,"Земля же была безвидна и пуста, и тьма над без..."
2,2,Бытие 1:3,И сказал Бог: да будет свет. И стал свет.


## 4. Кодирование корпуса

Каждый стих кодируем с префиксом `passage: ` и нормируем (для косинуса)

In [4]:
import numpy as np, torch
from sentence_transformers import SentenceTransformer

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Устройство:', DEVICE)

base_model = SentenceTransformer('intfloat/multilingual-e5-small', device=DEVICE)

EMB_PATH = f'{WORK}/emb_base.npy'
if os.path.exists(EMB_PATH):
    emb = np.load(EMB_PATH)
    print('Загружены готовые эмбеддинги:', emb.shape)
else:
    passages = ('passage: ' + corpus['verse']).tolist()
    emb = base_model.encode(
        passages, batch_size=256, normalize_embeddings=True,
        show_progress_bar=True, convert_to_numpy=True
    ).astype('float32')
    np.save(EMB_PATH, emb)
    print('Эмбеддинги посчитаны и сохранены:', emb.shape)

Устройство: cuda


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:124: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/387 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/498k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/57.0 [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/167 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/200 [00:00<?, ?B/s]

Batches:   0%|          | 0/145 [00:00<?, ?it/s]

Эмбеддинги посчитаны и сохранены: (37098, 384)


## 5. Индекс FAISS

`IndexFlatIP` по нормированным векторам = точный косинусный поиск.

In [5]:
import faiss

dim = emb.shape[1]                 # 384 у e5-small
index = faiss.IndexFlatIP(dim)
index.add(emb)
assert index.ntotal == len(corpus)

faiss.write_index(index, f'{WORK}/index_base.faiss')
print('Индекс построен и сохранён. Векторов:', index.ntotal, '| dim:', dim)

Индекс построен и сохранён. Векторов: 37098 | dim: 384


## 6. Функция поиска

Запрос кодируем с префиксом `query: `, нормируем, берём top-k.

Позиции из FAISS — это `verse_id`, по ним достаём метаданные из корпуса.

In [9]:
def search(query, k=5, model=base_model, idx=index):
    q = model.encode(['query: ' + query], normalize_embeddings=True,
                     convert_to_numpy=True).astype('float32')
    scores, ids = idx.search(q, k)
    out = corpus.iloc[ids[0]][['verse_id','reference','verse']].copy()
    out['score'] = scores[0]
    return out.reset_index(drop=True)

def show(query, k=5):
    print(f'Запрос: {query}\n' + '-'*70)
    for _, r in search(query, k).iterrows():
        print(f'{r.score:.3f}  [{r.reference}]  {r.verse}')
    print()

## 7. Дымовой тест

Проверяем, что поиск осмысленный ещё до дообучения.

Запросы намеренно не повторяют слова стихов — это проверка семантики.

In [10]:
for q in [
    'кто построил ковчег для спасения от потопа',
    'о силе любви, которая крепче смерти',
    'не тревожьтесь о завтрашнем дне',
    'свобода через познание',
    'что человек уносит с собой после смерти',
]:
    show(q, k=3)

Запрос: кто построил ковчег для спасения от потопа
----------------------------------------------------------------------
0.842  [К Евреям 11:7]  Верою Ной, получив откровение о том, что еще не было видимо, благоговея приготовил ковчег для спасения дома своего; ею осудил он (весь) мир, и сделался наследником праведности по вере.
0.837  [2-я Царств 6:3]  И поставили ковчег Божий на новую колесницу и вывезли его из дома Аминадава, что на холме. Сыновья же Аминадава, Оза и Ахио, вели новую колесницу.
0.835  [1-е Петра 3:20]  некогда непокорным ожидавшему их Божию долготерпению, во дни Ноя, во время строения ковчега, в котором немногие, то есть восемь душ, спаслись от воды.

Запрос: о силе любви, которая крепче смерти
----------------------------------------------------------------------
0.883  [Сирах 38:18]  ибо от печали бывает смерть, и печаль сердечная истощит силу.
0.880  [Сирах 51:8]  Душа моя близка была к смерти,
0.876  [Сирах 7:17]  Глубоко смири душу твою.

Запрос: не тревожьтесь

Baseline-ретривер работает и сохранён на Drive (`index_base.faiss`, `emb_base.npy`).

Baseline ведёт себя ровно так, как и должен слабый недообученный ретривер: он справляется, когда в запросе есть лексический мост к стиху («ковчег» -> Евр 11:7, «познание/истина» -> Ин 8:32), и проваливается на чистой семантике. «Сила любви крепче смерти» не находит Песнь 8:6, «не тревожьтесь о завтрашнем дне» не находит Мф 6:34, «что человек уносит после смерти» не находит Иов 1:21.

Видна и характерная аномалия — перекос в сторону Сираха и других длинных учительных книг: модель цепляется за отдельные токены («смерть», «печаль») и тянет длинные стихи.

## Фаза 4 — Метрики baseline

Recall@1/5/10 и MRR@10 на синтетическом `test` и ручном `gold`.

Функция оценки сама добавляет префикс `query: `, поэтому метрики не зависят
от правок в выводе `search`/`show`.

In [11]:
test_q = pd.read_csv(f'{WORK}/test_queries.csv')
gold_q = pd.read_csv(f'{WORK}/gold_manual.csv')
assert {'verse_id','query','query_type'} <= set(test_q.columns)
print('test:', len(test_q), '| gold:', len(gold_q))

test: 20 | gold: 29


In [12]:
import numpy as np

verse_ids = corpus['verse_id'].values

def evaluate(df_q, model, idx, ks=(1, 5, 10), max_k=10):
    qs = ('query: ' + df_q['query']).tolist()
    qemb = model.encode(qs, normalize_embeddings=True,
                        convert_to_numpy=True, batch_size=128).astype('float32')
    _, I = idx.search(qemb, max_k)
    ranked = verse_ids[I]
    gold = df_q['verse_id'].values[:, None]
    match = (ranked == gold)
    m = {f'Recall@{k}': match[:, :k].any(1).mean() for k in ks}
    ranks = match.argmax(1) + 1
    found = match.any(1)
    m[f'MRR@{max_k}'] = np.where(found, 1.0/ranks, 0.0).mean()
    return m, ranked, np.where(found, 1.0/ranks, 0.0)

In [13]:
res = {}
res['baseline | test'], _, _        = evaluate(test_q, base_model, index)
res['baseline | gold'], _, rr_gold  = evaluate(gold_q, base_model, index)
pd.DataFrame(res).T.round(3)

,Recall@1,Recall@5,Recall@10,MRR@10
baseline | test,0.050,0.100,0.150,0.082
baseline | gold,0.103,0.172,0.241,0.137


In [14]:
both = pd.concat([test_q, gold_q], ignore_index=True)
rows = []
for qt, grp in both.groupby('query_type'):
    m, _, _ = evaluate(grp, base_model, index)
    m['n'] = len(grp); m['type'] = qt; rows.append(m)
pd.DataFrame(rows).set_index('type').round(3)

,Recall@1,Recall@5,Recall@10,MRR@10,n
type,,,,,
factual,0.000,0.083,0.083,0.042,12
paraphrase,0.176,0.235,0.294,0.214,17
thematic,0.050,0.100,0.200,0.073,20


In [15]:
miss = gold_q[rr_gold == 0][['reference', 'query']]
print(f'Промахи baseline на gold: {len(miss)} из {len(gold_q)}')
for _, r in miss.iterrows():
    print(f'  [{r.reference}]  {r.query}')

Промахи baseline на gold: 22 из 29
  [Бытие 1:1]  С чего начинается творение мира
  [Исход 20:13]  Заповедь о неприкосновенности жизни
  [Иов 1:21]  Слова страдальца, потерявшего всё
  [Иов 1:21]  О том, что человек ничего не уносит с собой
  [Псалтирь 22:1]  Образ Бога как пастуха
  [Псалтирь 50:12]  Молитва об обновлении сердца после греха
  [Исаия 2:4]  Образ всеобщего мира без войны
  [Исаия 53:5]  Об исцелении через страдания одного
  [Даниил 5:25]  Загадочная надпись на стене пиршества
  [Иона 2:1]  Где пророк провёл три дня
  [От Матфея 5:3]  С чего начинаются заповеди блаженства
  [От Матфея 22:37]  Какая заповедь названа наибольшей
  [От Луки 10:30]  Притча о человеке, попавшем к разбойникам
  [От Иоанна 1:1]  Чем открывается четвёртое Евангелие
  [От Иоанна 3:16]  О даре Сына ради спасения мира
  [От Иоанна 14:6]  Как Христос определяет путь к Отцу
  [К Римлянам 6:23]  Что человек получает за грех и что — дар Божий
  [К Римлянам 8:28]  О том, что всё служит ко благу любящим Б

In [16]:
import json
json.dump({k: {m: float(v) for m, v in d.items()} for k, d in res.items()},
          open(f'{WORK}/metrics_baseline.json', 'w'),
          ensure_ascii=False, indent=2)
print('Сохранено: metrics_baseline.json')

Сохранено: metrics_baseline.json


## Fine-tuning эмбеддера и сравнение с baseline

Дообучаем multilingual-e5-small на train_queries с MultipleNegativesRankingLoss
(в качестве негативов — остальные стихи батча). Затем перекодируем корпус
дообученной моделью, строим новый FAISS-индекс и считаем те же метрики.

In [20]:
from sentence_transformers import InputExample

train_q = pd.read_csv(f'{WORK}/train_queries.csv')

dups = train_q['verse_id'].duplicated().sum()
print('train пар:', len(train_q), '| повторов verse_id:', dups)

train_examples = [
    InputExample(texts=['query: ' + r.query, 'passage: ' + r.verse])
    for r in train_q.itertuples()
]
print('InputExample готово:', len(train_examples))

train пар: 106 | повторов verse_id: 0
InputExample готово: 106


In [21]:
import math, torch
from torch.utils.data import DataLoader
from sentence_transformers import SentenceTransformer, losses

ft_model = SentenceTransformer('intfloat/multilingual-e5-small', device=DEVICE)

BATCH, EPOCHS = 16, 3
loader = DataLoader(train_examples, shuffle=True, batch_size=BATCH, drop_last=True)
loss = losses.MultipleNegativesRankingLoss(ft_model)
warmup = math.ceil(len(loader) * EPOCHS * 0.1)

ft_model.fit(
    train_objectives=[(loader, loss)],
    epochs=EPOCHS,
    warmup_steps=warmup,
    show_progress_bar=True,
)

FT_DIR = f'{WORK}/e5_finetuned'
ft_model.save(FT_DIR)
print('Модель дообучена и сохранена:', FT_DIR)

/tmp/ipykernel_1628/3077320515.py:3: DeprecationWarning: Importing from 'sentence_transformers.losses' is deprecated and will be removed in a future version. Please use 'sentence_transformers.sentence_transformer.losses' instead.
  from sentence_transformers import SentenceTransformer, losses


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Computing widget examples:   0%|          | 0/1 [00:00<?, ?example/s]

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 2, 'bos_token_id': 0, 'pad_token_id': 1}.


Step,Training Loss


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Модель дообучена и сохранена: /content/drive/MyDrive/semantic_bible/e5_finetuned


In [22]:
ft_passages = ('passage: ' + corpus['verse']).tolist()
ft_emb = ft_model.encode(ft_passages, batch_size=256, normalize_embeddings=True,
                         show_progress_bar=True, convert_to_numpy=True).astype('float32')
np.save(f'{WORK}/emb_ft.npy', ft_emb)

ft_index = faiss.IndexFlatIP(ft_emb.shape[1])
ft_index.add(ft_emb)
faiss.write_index(ft_index, f'{WORK}/index_ft.faiss')
print('Новый индекс построен. Векторов:', ft_index.ntotal)

Batches:   0%|          | 0/145 [00:00<?, ?it/s]

Новый индекс построен. Векторов: 37098


In [26]:
comp = {}
comp['base | test'],  _, _ = evaluate(test_q, base_model, index)
comp['ft   | test'],  _, _ = evaluate(test_q, ft_model,  ft_index)
comp['base | gold'],  _, rr_base_gold = evaluate(gold_q, base_model, index)
comp['ft   | gold'],  _, rr_ft_gold   = evaluate(gold_q, ft_model,  ft_index)

comp_df = pd.DataFrame(comp).T.round(3)
display(comp_df)

delta = (pd.Series(comp['ft   | gold']) - pd.Series(comp['base | gold'])).round(3)
print('Прирост на gold (ft - base):')
print(delta.to_string())

,Recall@1,Recall@5,Recall@10,MRR@10
base | test,0.050,0.100,0.150,0.082
ft | test,0.150,0.200,0.200,0.162
base | gold,0.103,0.172,0.241,0.137
ft | gold,0.138,0.241,0.345,0.183


Прирост на gold (ft - base):
Recall@1     0.034
Recall@5     0.069
Recall@10    0.103
MRR@10       0.046


In [27]:
fixed = gold_q[(rr_base_gold == 0) & (rr_ft_gold > 0)][['reference', 'query']]
print(f'Стихов, поднятых в топ-10 после дообучения: {len(fixed)} из 22 промахов baseline')
for _, r in fixed.iterrows():
    print(f'  [{r.reference}]  {r.query}')

print('\nПо типам запросов (test+gold), Recall@10 и MRR@10:')
rows = []
for qt, grp in pd.concat([test_q, gold_q]).groupby('query_type'):
    mb, _, _ = evaluate(grp, base_model, index)
    mf, _, _ = evaluate(grp, ft_model, ft_index)
    rows.append({'type': qt, 'n': len(grp),
                 'R@10 base': round(mb['Recall@10'], 3), 'R@10 ft': round(mf['Recall@10'], 3),
                 'MRR base': round(mb['MRR@10'], 3),    'MRR ft': round(mf['MRR@10'], 3)})
display(pd.DataFrame(rows).set_index('type'))

Стихов, поднятых в топ-10 после дообучения: 3 из 22 промахов baseline
  [От Луки 10:30]  Притча о человеке, попавшем к разбойникам
  [От Иоанна 14:6]  Как Христос определяет путь к Отцу
  [К Римлянам 8:28]  О том, что всё служит ко благу любящим Бога

По типам запросов (test+gold), Recall@10 и MRR@10:


,n,R@10 base,R@10 ft,MRR base,MRR ft
type,,,,,
factual,12,0.083,0.083,0.042,0.021
paraphrase,17,0.294,0.353,0.214,0.279
thematic,20,0.200,0.350,0.073,0.178


In [28]:
import json

def compare_search(query, k=3):
    print(f'Запрос: {query}\n' + '='*70)
    for tag, m, ix in [('BASE', base_model, index), ('FT  ', ft_model, ft_index)]:
        q = m.encode(['query: ' + query], normalize_embeddings=True,
                     convert_to_numpy=True).astype('float32')
        s, ids = ix.search(q, k)
        top = corpus.iloc[ids[0]]
        print(tag, '->', ' | '.join(f'[{r.reference}]' for _, r in top.iterrows()))
    print()

for q in ['образ Бога как пастуха',
          'кратчайшее определение Бога',
          'что человек уносит с собой после смерти',
          'надпись на стене во время пира']:
    compare_search(q)

json.dump({k: {m: float(v) for m, v in d.items()} for k, d in comp.items()},
          open(f'{WORK}/metrics_finetuned.json', 'w'), ensure_ascii=False, indent=2)
print('Сохранено: metrics_finetuned.json')

Запрос: образ Бога как пастуха
BASE -> [Псалтирь 49:2] | [Исаия 40:18] | [К Колоссянам 1:15]
FT   -> [Исаия 40:18] | [К Римлянам 1:23] | [Псалтирь 49:2]

Запрос: кратчайшее определение Бога
BASE -> [Сирах 11:21] | [2-я Царств 22:31] | [Сирах 11:22]
FT   -> [2-я Царств 22:31] | [Сирах 11:21] | [Иов 15:11]

Запрос: что человек уносит с собой после смерти
BASE -> [2-е Коринфянам 2:16] | [Левит 13:18] | [2-я Паралипоменон 21:15]
FT   -> [Левит 13:18] | [Сирах 38:23] | [2-е Коринфянам 2:16]

Запрос: надпись на стене во время пира
BASE -> [3-я Маккавейская 5:9] | [Неемия 13:21] | [Даниил 5:5]
FT   -> [От Иоанна 2:8] | [Исаия 25:5] | [Михей 7:11]

Сохранено: metrics_finetuned.json


## Генеративный слой (RAG)

Дообученный e5 извлекает топ-k стихов, Qwen2.5-7B-Instruct (4-бит) формирует
ответ СТРОГО по ним, со ссылками.

In [29]:
import gc, torch
for name in ['base_model', 'index', 'emb']:
    if name in globals():
        del globals()[name]
gc.collect(); torch.cuda.empty_cache()
print('Свободно на GPU:',
      round(torch.cuda.mem_get_info()[0]/1e9, 2), 'ГБ из',
      round(torch.cuda.mem_get_info()[1]/1e9, 2))

Свободно на GPU: 13.96 ГБ из 15.64


In [30]:
!pip -q install -U bitsandbytes accelerate transformers

from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

LLM_NAME = 'Qwen/Qwen2.5-7B-Instruct'
bnb = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)
tok = AutoTokenizer.from_pretrained(LLM_NAME)
llm = AutoModelForCausalLM.from_pretrained(
    LLM_NAME, quantization_config=bnb, device_map='auto', torch_dtype=torch.float16
)
print('LLM загружена. Занято на GPU:',
      round(torch.cuda.memory_allocated()/1e9, 2), 'ГБ')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 13.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.2/11.2 MB 99.9 MB/s eta 0:00:00


config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

...

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


generation_config.json:   0%|          | 0.00/243 [00:00<?, ?B/s]

LLM загружена. Занято на GPU: 6.53 ГБ


In [31]:
def retrieve(query, k=5):
    q = ft_model.encode(['query: ' + query], normalize_embeddings=True,
                        convert_to_numpy=True).astype('float32')
    scores, ids = ft_index.search(q, k)
    out = corpus.iloc[ids[0]][['reference', 'verse']].copy()
    out['score'] = scores[0]
    return out.reset_index(drop=True)

In [37]:
SYSTEM = (
    'Ты помощник по тексту Синодального перевода Библии. '
    'Отвечай на вопрос ТОЛЬКО на основе приведённых стихов. '
    'Не добавляй сведений извне и не домысливай. '
    'Обязательно ссылайся на стихи в формате [Книга глава:стих]. '
    'Если в приведённых стихах нет ответа, прямо скажи об этом.'
)

def build_context(hits):
    return '\n'.join(f'[{r.reference}] {r.verse}' for r in hits.itertuples())

def rag_answer(question, k=5, max_new_tokens=300, show_sources=True):
    hits = retrieve(question, k)
    context = build_context(hits)
    messages = [
        {'role': 'system', 'content': SYSTEM},
        {'role': 'user',
         'content': f'Стихи:\n{context}\n\nВопрос: {question}'},
    ]
    inputs = tok.apply_chat_template(
        messages, add_generation_prompt=True,
        return_tensors='pt', return_dict=True
    ).to(llm.device)
    out = llm.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)
    answer = tok.decode(out[0, inputs['input_ids'].shape[1]:],
                        skip_special_tokens=True).strip()

    if show_sources:
        print('Найденные стихи:')
        for r in hits.itertuples():
            print(f'  {r.score:.3f}  [{r.reference}]  {r.verse[:80]}')
        print('\nОтвет:\n' + answer + '\n' + '='*70)
    return answer

In [38]:
import warnings; warnings.filterwarnings('ignore').
for q in [
    'Что Писание говорит о любви к врагам?',
    'Кто такой добрый пастырь?',
    'Что человек уносит с собой после смерти?',
    'Какая заповедь названа наибольшей?',
    'Как звали президента России?',   # проверяем отказ от домысливания
]:
    print('ВОПРОС:', q)
    rag_answer(q, k=5)

ВОПРОС: Что Писание говорит о любви к врагам?


/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


Найденные стихи:
  0.806  [К Римлянам 11:28]  В отношении к благовестию, они враги ради вас; а в отношении к избранию, возлюбл
  0.803  [К Римлянам 8:35]  Кто отлучит нас от любви Божией: скорбь, или теснота, или гонение, или голод, ил
  0.798  [От Луки 6:27]  Но вам, слушающим, говорю: любите врагов ваших, благотворите ненавидящим вас,
  0.792  [От Матфея 5:44]  А Я говорю вам: любите врагов ваших, благословляйте проклинающих вас, благотвори
  0.791  [Псалтирь 96:10]  Любящие Господа, ненавидьте зло! Он хранит души святых Своих; из руки нечестивых

Ответ:
Писание говорит о том, что следует любить даже тех, кто враждебен вам. Это указывается в нескольких местах:

1. [От Луки 6:27] — Иисус говорит: "Но вам, слушающим, говорю: любите врагов ваших, благотворите ненавидящим вас,"
2. [От Матфея 5:44] — Иисус повторяет это учение: "А Я говорю вам: любите врагов ваших, благословляйте проклинающих вас, благотворите ненавидящим вас и молитесь за обижающих вас и гонящих вас,"

Эти стихи показыва

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


Найденные стихи:
  0.820  [От Иоанна 10:11]  Я есмь пастырь добрый: пастырь добрый полагает жизнь свою за овец.
  0.795  [От Иоанна 10:14]  Я есмь пастырь добрый; и знаю Моих, и Мои знают Меня.
  0.781  [Иеремия 10:21]  ибо пастыри сделались бессмысленными и не искали Господа, а потому они и поступа
  0.771  [1-я Паралипоменон 4:40]  и нашли пастбища тучные и хорошие и землю обширную, спокойную и безопасную, пото
  0.768  [1-е Петра 5:1]  Пастырей ваших умоляю я, сопастырь и свидетель страданий Христовых и соучастник 

Ответ:
Добрый пастырь упоминается в стихе [От Иоанна 10:11], где сказано: "Я есмь пастырь добрый: пастырь добрый полагает жизнь свою за овец."
ВОПРОС: Что человек уносит с собой после смерти?


...

Найденные стихи:
  0.824  [2-е Коринфянам 2:16]  для одних запах смертоносный на смерть, а для других запах живительный на жизнь.
  0.817  [Левит 13:18]  Если у кого на коже тела был нарыв и зажил,
  0.810  [К Римлянам 7:24]  Бедный я человек! кто избавит меня от сего тела смерти?
  0.808  [Иезекииль 44:25]  К мертвому человеку никто из них не должен подходить, чтобы не сделаться нечисты
  0.807  [Сирах 10:13]  Когда же человек умрет, то наследием его становятся пресмыкающиеся, звери и черв

Ответ:
По данным приведённых стихов, не указано, что человек уносит с собой после смерти. Ближайший по теме стих — это [Сирах 10:13], который говорит о том, что когда человек умрет, его наследием становятся пресмыкающиеся, звери и черви, но это относится к наследию в буквальном смысле слова, а не к чему-то, что человек уносит с собой.
ВОПРОС: Какая заповедь названа наибольшей?


/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


Найденные стихи:
  0.863  [От Матфея 22:36]  Учитель! какая наибольшая заповедь в законе?
  0.830  [От Матфея 22:38]  сия есть первая и наибольшая заповедь;
  0.819  [Исаия 28:10]  Ибо все заповедь на заповедь, заповедь на заповедь, правило на правило, правило 
  0.787  [К Римлянам 13:9]  Ибо заповеди: не прелюбодействуй, не убивай, не кради, не лжесвидетельствуй, не 
  0.782  [От Марка 10:19]  Знаешь заповеди: не прелюбодействуй, не убивай, не кради, не лжесвидетельствуй, 

Ответ:
Наибольшая заповедь, согласно указанным стихам, названа как "сия есть первая и наибольшая заповедь" ([От Матфея 22:38]). Однако конкретный текст этой самой наибольшей заповеди не приведен в данных стихах.
ВОПРОС: Как звали президента России?


/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


Найденные стихи:
  0.765  [Откровение Иоанна Богослова 19:16]  На одежде и на бедре Его написано имя: "Царь царей и Господь господствующих".
  0.754  [4-я Царств 18:18]  И звали они царя. И вышел к ним Елиаким, сын Хелкиин, начальник дворца, и Севна 
  0.746  [Иона 3:6]  Это слово дошло до царя Ниневии, и он встал с престола своего, и снял с себя цар
  0.741  [От Марка 3:16]  поставил Симона, нарекши ему имя Петр,
  0.740  [Исаия 9:6]  Ибо младенец родился нам - Сын дан нам; владычество на раменах Его, и нарекут им

Ответ:
В приведённых стихах нет информации о президенте России или его имени.


## Демо на Gradio

Веб-форма поверх готовой RAG-цепочки: вопрос -> дообученный e5 находит топ-k
стихов -> Qwen формирует ответ со ссылками. Ползунок k регулирует число стихов
в контексте. Публичная ссылка (share=True) действует 72 часа.

In [43]:
import warnings; warnings.filterwarnings('ignore')

def rag_for_ui(question, k, progress=gr.Progress()):
    question = (question or '').strip()
    if not question:
        yield 'Введите вопрос.', ''
        return

    # Стадия 1: поиск стихов
    progress(0.2, desc='Ищу стихи...')
    hits = retrieve(question, int(k))
    context = build_context(hits)
    sources = '\n'.join(
        f'**[{r.reference}]** _(score {r.score:.3f})_  \n{r.verse}'
        for r in hits.itertuples()
    )
    # сразу показываем стихи, ответ пока генерируется
    yield '_Генерирую ответ..._', sources

    # Стадия 2: генерация ответа
    progress(0.6, desc='Генерирую ответ...')
    messages = [
        {'role': 'system', 'content': SYSTEM},
        {'role': 'user', 'content': f'Стихи:\n{context}\n\nВопрос: {question}'},
    ]
    inputs = tok.apply_chat_template(
        messages, add_generation_prompt=True,
        return_tensors='pt', return_dict=True
    ).to(llm.device)
    out = llm.generate(**inputs, max_new_tokens=300, do_sample=False)
    answer = tok.decode(out[0, inputs['input_ids'].shape[1]:],
                        skip_special_tokens=True).strip()

    yield answer, sources

In [44]:
!pip -q install gradio
import gradio as gr

EXAMPLES = [
    'Что Писание говорит о любви к врагам?',
    'Кто такой добрый пастырь?',
    'Что сказано о вере и делах?',
    'Какая заповедь названа наибольшей?',
]

with gr.Blocks(title='Semantic Bible — RAG по Синодальному переводу') as demo:
    gr.Markdown('## Semantic Bible\nПоиск и ответы по тексту Синодального '
                'перевода Библии (дообученный e5 + FAISS + Qwen2.5).')
    with gr.Row():
        with gr.Column(scale=3):
            q = gr.Textbox(label='Вопрос', placeholder='Например: что сказано о прощении?')
        with gr.Column(scale=1):
            k = gr.Slider(1, 10, value=5, step=1, label='Стихов в контексте (k)')
    btn = gr.Button('Найти и ответить', variant='primary')
    answer = gr.Markdown(label='Ответ')
    gr.Markdown('### Найденные стихи')
    sources = gr.Markdown()

    btn.click(rag_for_ui, inputs=[q, k], outputs=[answer, sources])
    q.submit(rag_for_ui, inputs=[q, k], outputs=[answer, sources])
    gr.Examples(EXAMPLES, inputs=q)

demo.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://7776766af361f2b008.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
